# **Importações e configuração**

In [ ]:
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import random
import re
import nltk
import torch
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from wordcloud import WordCloud, STOPWORDS
from collections import Counter
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_fscore_support, accuracy_score
from transformers import pipeline
from tqdm import tqdm

# ============================
# Downloads necessários do NLTK
# ============================
# Esses downloads garantem que os recursos de stopwords, lematização e tokenização
# estejam disponíveis para processamento de texto.
nltk.download("stopwords")                       # Lista de palavras irrelevantes (stopwords)
nltk.download("wordnet")                         # Base de dados para lematização
nltk.download("omw-1.4")                         # Traduções e suporte adicional ao WordNet
nltk.download("punkt")                           # Tokenizador de frases e palavras
nltk.download("punkt_tab")                       # Dados adicionais para tokenização
nltk.download("averaged_perceptron_tagger_eng")  # Versão em inglês do POS tagging

# ============================
# Configuração de stopwords e lematizador
# ============================
stop_words = set(stopwords.words("english"))     # Define conjunto de palavras irrelevantes em inglês
lemmatizer = WordNetLemmatizer()                 # Inicializa o lematizador para reduzir palavras à forma base



# **Carregar base de dados**

In [ ]:
# ============================
# Carregamento e preparação dos dados
# ============================

# Lê o arquivo CSV contendo os dados já processados de reclamações
df = pd.read_csv("complaints_processed.csv")

# Remove colunas que foram criadas automaticamente pelo pandas (ex.: "Unnamed: 0")
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Remove linhas onde a coluna "narrative" está vazia (NaN)
df = df.dropna(subset=["narrative"])

# Remove linhas onde "narrative" é apenas espaço em branco
df = df[df["narrative"].str.strip() != ""]

# Cria uma amostra aleatória de 60.000 registros para trabalhar,
# garantindo reprodutibilidade com random_state=42
df_sample = df.sample(60000, random_state=42).copy()


# **Pré-processamento**

In [ ]:
# ============================
# Função auxiliar para mapear tags do POS (part-of-speech) para WordNet
# ============================
def get_wordnet_pos(tag):
    # Se a tag começar com "J", é um adjetivo
    if tag.startswith("J"):
        return wordnet.ADJ
    # Se começar com "V", é um verbo
    elif tag.startswith("V"):
        return wordnet.VERB
    # Se começar com "N", é um substantivo
    elif tag.startswith("N"):
        return wordnet.NOUN
    # Se começar com "R", é um advérbio
    elif tag.startswith("R"):
        return wordnet.ADV
    # Caso contrário, assume substantivo como padrão
    else:
        return wordnet.NOUN

# ============================
# Função de limpeza de texto usando NLTK
# ============================
def clean_text_nltk(text):
    # Converte para string, coloca em minúsculas e remove espaços extras
    text = str(text).lower().strip()

    # Remove sequências de "x" repetidas (ex.: "xxxx")
    text = re.sub(r"x{2,}", " ", text)

    # Tokeniza o texto em palavras
    tokens = word_tokenize(text)

    # Mantém apenas tokens alfabéticos e que não sejam stopwords
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]

    # Aplica POS tagging (identificação da classe gramatical de cada palavra)
    tagged = pos_tag(tokens)

    # Lematiza cada palavra de acordo com sua classe gramatical
    tokens = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in tagged]

    # Junta os tokens novamente em uma string limpa
    return " ".join(tokens)

# ============================
# Aplicação da limpeza no dataset
# ============================
# Cria uma nova coluna "clean_text" com os textos já pré-processados
df_sample["clean_text"] = df_sample["narrative"].apply(clean_text_nltk)



# Criação da coluna sentiment e classificação com DistilBERT

In [ ]:
# ============================
# Carregar pipeline de análise de sentimento com DistilBERT
# ============================

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    return_all_scores=True
)

# ============================
# Função para processar textos em lotes
# ============================
def sentiment_batch(texts, max_length=512, batch_size=64):
    results = []
    tokenizer = sentiment_pipeline.tokenizer

    for i in tqdm(range(0, len(texts), batch_size), desc="Processando lotes"):
        batch = texts[i:i+batch_size]
        batch_chunks = []

        for text in batch:
            tokens = tokenizer.encode(str(text), add_special_tokens=False)
            if len(tokens) <= max_length:
                batch_chunks.append([text])
            else:
                chunk_texts = []
                for j in range(0, len(tokens), max_length):
                    chunk_tokens = tokens[j:j+max_length]
                    chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
                    chunk_texts.append(chunk_text)
                batch_chunks.append(chunk_texts)

        flat_chunks = [chunk for chunks in batch_chunks for chunk in chunks]
        chunk_results = sentiment_pipeline(flat_chunks, truncation=True, max_length=max_length)

        idx = 0
        for chunks in batch_chunks:
            chunk_labels = []
            for _ in chunks:
                scores_for_current_chunk = chunk_results[idx]
                if isinstance(scores_for_current_chunk, dict):
                    scores_for_current_chunk = [scores_for_current_chunk]
                pos_score = next((s['score'] for s in scores_for_current_chunk if s['label'] == 'POSITIVE'), 0.0)
                neg_score = next((s['score'] for s in scores_for_current_chunk if s['label'] == 'NEGATIVE'), 0.0)
                chunk_labels.append("POSITIVE" if pos_score >= neg_score else "NEGATIVE")
                idx += 1
            majority_label = max(set(chunk_labels), key=chunk_labels.count)
            results.append(majority_label)

    return results

# ============================
# Aplicar análise de sentimento
# ============================
texts = df_sample.head(20000)["clean_text"].tolist()
df_sample_distil = df_sample.head(20000).copy()
df_sample_distil["sentiment"] = sentiment_batch(texts, batch_size=64)

# Exibe a contagem de rótulos POSITIVE / NEGATIVE
print(df_sample_distil["sentiment"].value_counts())

# ============================
# Salvamento do dataset em CSV e Pickle
# ============================
# Salvar em CSV
df_sample_distil.to_csv("dataset_com_sentimento.csv", index=False)

# Salvar em Pickle
df_sample_distil.to_pickle("dataset_com_sentimento.pkl")

print("Dataset salvo em 'dataset_com_sentimento.csv' e 'dataset_com_sentimento.pkl'")


In [ ]:
# Carregar o CSV salvo
df_sample_distil = pd.read_csv("dataset_com_sentimento.csv")

# Visualizar as primeiras linhas
df_sample_distil.head()


# **Treinamento de Modelo (TF-IDF + Keras)**

In [ ]:
# ============================
# Listas expandidas automaticamente
# ============================
subjects = [
    "My credit card", "The checking account", "This loan option",
    "The savings account", "Customer service", "The mortgage process",
    "My financing plan", "The personal loan", "My current account",
    "My debit card", "The online account", "The investment plan",
    "The banking app", "The financial advisor", "The payment service"
]

adjectives = [
    "excellent", "fantastic", "reliable", "convenient",
    "transparent", "outstanding", "professional", "helpful", "secure",
    "amazing", "efficient", "trustworthy", "remarkable", "friendly",
    "innovative", "supportive", "clear", "accessible"
]

experiences = [
    "everything worked smoothly", "the approval was quick",
    "the rewards exceeded expectations", "the staff was very supportive",
    "the process was simple", "I feel very satisfied",
    "it was handled efficiently", "I highly recommend it",
    "the digital experience was seamless", "the guidance was reassuring",
    "the response time was impressive", "I felt valued as a customer",
    "the instructions were easy to follow", "the outcome exceeded my hopes",
    "the service was beyond expectations", "I would choose it again"
]

# ============================
# Função otimizada para gerar comentários positivos únicos
# ============================
def generate_unique_positive_comments(n):
    all_combinations = [
        f"{s} is {a}, {e}."
        for s, a, e in itertools.product(subjects, adjectives, experiences)
    ]
    if n > len(all_combinations):
        base = random.sample(all_combinations, len(all_combinations))
        extra_needed = n - len(all_combinations)
        extras = [
            f"{comment} (extra {i})"
            for i, comment in enumerate(random.choices(all_combinations, k=extra_needed), 1)
        ]
        return base + extras
    else:
        return random.sample(all_combinations, n)

# ============================
# Pipeline de balanceamento dinâmico
# ============================

# proporção alvo dinâmica entre 0.20 e 0.25
target_ratio = random.uniform(0.20, 0.25)

counts = df_sample_distil["sentiment"].value_counts()
pos = counts.get("POSITIVE", 0)
neg = counts.get("NEGATIVE", 0)

# cálculo correto para n_missing
n_missing = int(((target_ratio * (pos + neg)) - pos) / (1 - target_ratio))

# garante que não seja negativo
n_missing = max(0, n_missing)

# gera novos comentários
new_comments = generate_unique_positive_comments(n_missing)

df_new = pd.DataFrame({
    "narrative": new_comments,
    "sentiment": ["POSITIVE"] * len(new_comments)
})

df_balanced = pd.concat([df_sample_distil, df_new], ignore_index=True)

# ============================
# Conferir resultado
# ============================
abs_counts = df_balanced["sentiment"].value_counts()
perc_counts = (df_balanced["sentiment"].value_counts(normalize=True) * 100).round(2)

result = pd.DataFrame({
    "Absoluto": abs_counts,
    "Percentual (%)": perc_counts
})

print(f"Proporção alvo escolhida: {target_ratio:.2f}")
print(result)

In [ ]:
# Divisão em treino e teste
X_train_texts, X_test_texts, y_train, y_test = train_test_split(
    df_balanced["narrative"],
    df_balanced["sentiment"].map({"POSITIVE":1, "NEGATIVE":0}),
    test_size=0.2,
    stratify=df_balanced["sentiment"],
    random_state=42
)

# Vetorização com TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)

# Ajusta apenas nos dados de treino
X_train = vectorizer.fit_transform(X_train_texts)

# Aplica transformação nos dados de teste
X_test = vectorizer.transform(X_test_texts)

# Converter para arrays densos (necessário para Keras)
X_train_dense = X_train.toarray().astype(np.float32)
X_test_dense = X_test.toarray().astype(np.float32)

# Definição do modelo
def build_model(input_dim):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(128, activation="relu", input_shape=(input_dim,)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

# Treinamento
model = build_model(X_train_dense.shape[1])
history = model.fit(
    X_train_dense, y_train,
    epochs=5, batch_size=32,
    validation_data=(X_test_dense, y_test),
    verbose=1
)




# **Avaliação do modelo**

In [ ]:
# Obtém as probabilidades previstas pelo modelo para os dados de teste
pred_proba = model.predict(X_test_dense).ravel()

# Converte as probabilidades em previsões binárias (0 ou 1) usando o limiar de 0.5
y_pred = (pred_proba >= 0.5).astype(int)

# Exibe o relatório de classificação com métricas como precisão, recall e F1-score
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred, digits=4))

# Calcula e imprime a métrica AUC (Área sob a curva ROC), que avalia a qualidade da classificação
print("AUC:", roc_auc_score(y_test, pred_proba))


# **Análise das Dores dos clientes(Wordclouds e Frequência)**

In [ ]:
# ============================
# Preparação dos dados
# ============================
df_strat = df_balanced.copy()
# Cria uma cópia do dataframe balanceado para não alterar o original
df_strat["sentiment"] = df_strat["sentiment"].map({"POSITIVE":1, "NEGATIVE":0})
# Converte os rótulos de sentimento em valores numéricos (1 = positivo, 0 = negativo)

# ============================
# Wordcloud geral
# ============================
stopwords = set(STOPWORDS)  # Define palavras irrelevantes que serão removidas
neg_texts_all = " ".join(df_strat[df_strat["sentiment"]==0]["narrative"])
# Junta todos os textos negativos em uma única string

wc_all = WordCloud(width=1200, height=600,
                   background_color="white",
                   colormap="viridis",
                   stopwords=stopwords,
                   max_words=50,          # limita a 50 palavras mais frequentes
                   min_font_size=12).generate(neg_texts_all)

plt.figure(figsize=(12,10))  # Define tamanho da figura
plt.imshow(wc_all, interpolation="bilinear")  # Renderiza a nuvem de palavras
plt.axis("off")  # Remove eixos
plt.title("Principais termos em reclamações (geral)", fontsize=16)
plt.show()

# Espaço extra entre gráficos
print("\n\n")   # Insere duas quebras de linha

# ============================
# Wordcloud por produto (sem subplots)
# ============================
products = df_strat["product"].value_counts().index.tolist()[:6]
# Seleciona os 6 produtos mais frequentes

for prod in products:
    neg_texts = " ".join(
        df_strat[(df_strat["product"]==prod) & (df_strat["sentiment"]==0)]["narrative"]
    )
    if len(neg_texts) < 50:  # Ignora produtos com poucos textos
        continue

    wc = WordCloud(width=1200, height=600,
                   background_color="white",
                   colormap="viridis",
                   stopwords=stopwords,
                   max_words=50,
                   min_font_size=12).generate(neg_texts)

    plt.figure(figsize=(12,10))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Principais termos em {prod}", fontsize=16)
    plt.show()

    # Espaço extra entre gráficos
    print("\n\n")
    # Se quiser uma linha separadora mais visível, pode usar:
    # print("="*80)

# ============================
# Gráfico de frequência dos termos negativos por produto
# ============================
products = df_strat["product"].value_counts().index.tolist()[:6]
fig, axes = plt.subplots(len(products), 1, figsize=(12, 5*len(products)))
# Cria subplots, um para cada produto

for i, prod in enumerate(products):
    texts = " ".join(
        df_strat[(df_strat["product"] == prod) & (df_strat["sentiment"] == 0)]["narrative"]
    ).split()

    if len(texts) == 0:  # Se não houver textos, pula
        continue

    freq = Counter(texts).most_common(10)  # Seleciona os 10 termos mais comuns
    terms, counts = zip(*freq)

    sns.barplot(
        x=list(counts),
        y=list(terms),
        ax=axes[i],
        hue=list(terms),        # Usa os termos como "hue" para diferenciar cores
        palette="Blues_r",
        legend=False            # Remove legenda
    )

    axes[i].set_title(f"Top 10 termos em {prod}", fontsize=14)
    axes[i].set_xlabel("Frequência")
    axes[i].set_ylabel("Termos")

    # Adiciona valores numéricos nas barras
    for p in axes[i].patches:
        axes[i].annotate(f"{int(p.get_width())}",
                         (p.get_x() + p.get_width(), p.get_y() + p.get_height()/2),
                         ha="left", va="center", fontsize=10)

plt.tight_layout(pad=10)  # Ajusta espaçamento entre gráficos
plt.show()


# **Mapeamento de Temas**

In [ ]:
# ============================
# Clusterização inicial com KMeans
# ============================
n_clusters = 6

# Vetorização dos textos negativos
X = vectorizer.transform(df_strat[df_strat["sentiment"] == 0]["narrative"])

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X)
df_strat.loc[df_strat["sentiment"]==0, "cluster"] = clusters

# ============================
# Extração dos termos representativos por cluster
# ============================
terms = vectorizer.get_feature_names_out()
cluster_term_matrix = np.zeros((n_clusters, len(terms)))

for i in range(n_clusters):
    cluster_docs = X[clusters==i]
    cluster_mean = cluster_docs.sum(axis=0).A1
    cluster_term_matrix[i] = cluster_mean

# ============================
# Agrupamento dos clusters em blocos de temas
# ============================
n_blocos = 6  # igual ao número de clusters
kmeans_blocos = KMeans(n_clusters=n_blocos, random_state=42, n_init=10)
blocos = kmeans_blocos.fit_predict(cluster_term_matrix)

# ============================
# Criação automática de dicionário de termos por bloco
# ============================
bloco_terms = {}
for i in range(n_blocos):
    indices = np.where(blocos == i)[0]
    bloco_mean = cluster_term_matrix[indices].sum(axis=0)
    top_idx = bloco_mean.argsort()[-10:][::-1]  # pega os 10 termos mais fortes
    bloco_terms[i] = [terms[j] for j in top_idx]

# Dicionário com os termos mais frequentes
dicionario_blocos = {f"Bloco {i}": termos for i, termos in bloco_terms.items()}

# Exibe o dicionário para análise
print("Dicionário automático de termos por bloco:")
for bloco, termos in dicionario_blocos.items():
    print(f"{bloco}: {', '.join(termos)}")

# ============================
# Mapeamento dos temas
# ============================

mapa_temas = {
    "Bloco 0": "Problemas Conta Corrente e Cartão",
    "Bloco 1": "Fraude e Roubo de Identidade",
    "Bloco 2": "Pagamentos e Empréstimos Atrasados",
    "Bloco 3": "Relatórios de Crédito Incorretos",
    "Bloco 4": "Cobrança de Dívidas",
    "Bloco 5": "Disputas e Respostas Inadequadas"
}

# ============================
# Frequência total por bloco
# ============================
bloco_freq = []
for i in range(n_blocos):
    indices = np.where(blocos == i)[0]
    bloco_freq.append(cluster_term_matrix[indices].sum())

# Ordena blocos por frequência
ordem = np.argsort(bloco_freq)[::-1]
top_blocos = [mapa_temas.get(f"Bloco {i}", "Outros") for i in ordem]
top_valores = [bloco_freq[i] for i in ordem]

# ============================
# Gráfico de barras horizontais dos blocos
# ============================
plt.figure(figsize=(10,6))
sns.barplot(x=top_valores[:n_blocos], y=top_blocos[:n_blocos], palette="Blues_r", orient="h")
plt.title("Top Reclamações Bancárias")
plt.xlabel("Frequência absoluta")
plt.ylabel("Assuntos")

for i, val in enumerate(top_valores[:n_blocos]):
    plt.text(val + 0.5, i, f"{int(val)}", va="center")

plt.tight_layout()
plt.show()